# Breast Cancer Wisconsin — 能量泛函 QUBO
## 7×7 网格 FEM + Energy-Form QUBO + CIM 单次真机求解
PCA 降维 → 2D Poisson → $E(u)=\frac{1}{2}u^T K u - u^T F$ → CIM

In [1]:
import numpy as np
import pandas as pd
import kaiwu as kw
import warnings, json, os
warnings.filterwarnings('ignore')
kw.common.CheckpointManager.save_dir = '/tmp'
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

BIT_WIDTH = 8
OUTPUT_DIR = 'D:/QPDE/pde+pinn/outputs_bc'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('模块加载完成')

模块加载完成


In [2]:
# ===== 加载 Breast Cancer Wisconsin 数据 =====
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data'
cols = ['id','clump_thickness','cell_size','cell_shape','marginal_adhesion','epithelial_size',
        'bare_nuclei','bland_chromatin','normal_nucleoli','mitoses','class']
df = pd.read_csv(url, names=cols, na_values='?')
df = df.dropna().drop(columns=['id'])
X = df.drop(columns=['class']).values.astype(float)
y_raw = df['class'].values  # 2=benign, 4=malignant
y = np.where(y_raw == 4, 1, -1)  # malignant=+1, benign=-1
print(f'数据: {X.shape}, 正类(malignant): {(y==1).sum()}, 负类(benign): {(y==-1).sum()}')

数据: (683, 9), 正类(malignant): 239, 负类(benign): 444


In [3]:
# ===== PCA 降维到 2D + 缩放到 [0.15, 0.85] =====
X_scaled = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2).fit_transform(X_scaled)
x_min, x_max = X_2d[:,0].min(), X_2d[:,0].max()
y_min, y_max = X_2d[:,1].min(), X_2d[:,1].max()
X_2d[:,0] = (X_2d[:,0] - x_min) / (x_max - x_min) * 0.7 + 0.15
X_2d[:,1] = (X_2d[:,1] - y_min) / (y_max - y_min) * 0.7 + 0.15

# ===== 7x7 均匀网格 =====
N = 7; h = 1.0 / (N - 1)
grid_x = np.linspace(0, 1, N); grid_y = np.linspace(0, 1, N)
GX, GY = np.meshgrid(grid_x, grid_y, indexing='ij')
nodes_2d = np.column_stack([GX.ravel(), GY.ravel()])  # 49 nodes, order: i + j*N
boundary = (GX.ravel()==0) | (GX.ravel()==1) | (GY.ravel()==0) | (GY.ravel()==1)
internal = ~boundary
print(f'网格: {N}x{N}={N*N} 节点, 内部={internal.sum()}, 边界={boundary.sum()}')

网格: 7x7=49 节点, 内部=25, 边界=24


In [4]:
# ===== 高斯核源项构造: f(node) = Σ y_k * exp(-||node - p_k||² / (2σ²)) =====
sigma = 0.12
f = np.zeros(N*N)
for idx in range(len(y)):
    dist2 = np.sum((nodes_2d - X_2d[idx])**2, axis=1)
    f += y[idx] * np.exp(-dist2 / (2*sigma**2))

# ===== FD 5-point 拉普拉斯刚度矩阵 (内部节点) =====
idx_map = -np.ones(N*N, dtype=int)
idx_map[internal] = np.arange(internal.sum())  # global → internal local
n_i = internal.sum()
K_ii = np.zeros((n_i, n_i))
for i in range(1, N-1):
    for j in range(1, N-1):
        k = i * N + j
        ki = idx_map[k]
        K_ii[ki, ki] = 4.0
        for di, dj in [(-1,0),(1,0),(0,-1),(0,1)]:
            ni, nj = i+di, j+dj
            nk = ni * N + nj
            if internal[nk]:
                K_ii[ki, idx_map[nk]] = -1.0
            # 边界节点: K_ib * u_b = 0 (Dirichlet u=0), 不贡献 rhs

rhs = h**2 * f[internal]
print(f'K_ii: {K_ii.shape}, κ(K_ii)={np.linalg.cond(K_ii):.2e}, SPD={np.allclose(K_ii, K_ii.T)}')

K_ii: (25, 25), κ(K_ii)=1.39e+01, SPD=True


In [5]:
# ===== 经典参考解 (获取变量上下界) =====
u_ref = np.linalg.solve(K_ii, rhs)
margin = 0.5; rng = max(u_ref.max()-u_ref.min(), 0.1)
lb = u_ref.min() - margin * rng
ub = u_ref.max() + margin * rng
print(f'参考解范围: [{u_ref.min():.4f}, {u_ref.max():.4f}], 编码界: [{lb:.4f}, {ub:.4f}]')

参考解范围: [-5.0013, 1.2429], 编码界: [-8.1234, 4.3650]


In [6]:
# ===== 能量泛函 QUBO: E(u) = (1/2)u^T K_ii u - u^T rhs, Kronecker 构建 =====
def build_energy_qubo(K, r, bw, lb, ub):
    n = K.shape[0]; nvar = n * bw
    scale = (ub - lb) / (2**bw - 1)
    s = scale * np.array([2**k for k in range(bw)])
    ssT = np.outer(s, s); c = lb * np.ones(n); w = K @ c - r
    Q = np.zeros((nvar, nvar))
    for i in range(n):
        for j in range(i, n):
            aij = K[i,j]
            if abs(aij) < 1e-15: continue
            ri, rj = i*bw, j*bw
            blk = 0.5 * aij * ssT
            Q[ri:ri+bw, rj:rj+bw] += blk
            if i != j: Q[rj:rj+bw, ri:ri+bw] += blk.T
    for i in range(n):
        Q[i*bw:(i+1)*bw, i*bw:(i+1)*bw] += np.diag(w[i] * s)
    return Q.astype(np.float32), nvar, scale

Q_float, nvar, scale = build_energy_qubo(K_ii, rhs, BIT_WIDTH, lb, ub)
Q_qubo = kw.qubo.adjust_qubo_matrix_precision(Q_float)
print(f'QUBO: {nvar}x{nvar}, 值范围: [{Q_qubo.min():.1f}, {Q_qubo.max():.1f}]')

QUBO: 200x200, 值范围: [-376.0, 1024.0]


In [7]:
# ====== CIM 提交 (solve #1) ======
ising_mat, ising_bias = kw.conversion.qubo_matrix_to_ising_matrix(Q_qubo)
vars_ = [f'x[{i}]' for i in range(ising_mat.shape[0])]
ising_model = kw.ising.IsingModel(variables=vars_, ising_matrix=ising_mat, bias=ising_bias)
opt = kw.cim.CIMOptimizer(task_name='bc_energy', task_mode='quota')
opt.solve(ising_model.get_matrix())
print(f'bc_energy 已提交, Ising: {ising_mat.shape}')

[2026-05-20 18:10:12] [INFO    ] [kaiwu.cim._optimizer_adapter:6] - Task submit successfully, waiting for data validation. Task name: bc_energy
bc_energy 已提交, Ising: (201, 201)


In [8]:
# ====== CIM 取回 + 解码 (solve #2) ======
sol = opt.solve(ising_model.get_matrix())
print(f'返回: {sol.shape}')
sols_bin = (sol[:,:-1] * sol[:,-1:]+1)/2
energies = np.array([z@Q_qubo@z for z in sols_bin])
z_best = sols_bin[np.argmin(energies)]
u_quantum_i = np.zeros(n_i)
s = scale * np.array([2**k for k in range(BIT_WIDTH)])
for i in range(n_i):
    zi = z_best[i*BIT_WIDTH:(i+1)*BIT_WIDTH]
    u_quantum_i[i] = np.dot(s, zi) + lb
print(f'量子内部解: [{u_quantum_i.min():.4f}, {u_quantum_i.max():.4f}], 最优能量: {energies.min():.1f}')

[2026-05-20 18:12:21] [INFO    ] [kaiwu.cim._optimizer_adapter:2] - Task completed: bc_energy
返回: (10, 201)
量子内部解: [-5.0380, 0.2512], 最优能量: -6292.0


In [9]:
# ====== 保存预设解 ======
u_quantum = np.zeros(N*N); u_quantum[internal] = u_quantum_i
np.save(f'{OUTPUT_DIR}/preset_nodes_energy.npy', nodes_2d)
np.save(f'{OUTPUT_DIR}/preset_values_energy.npy', u_quantum)
np.save(f'{OUTPUT_DIR}/X_2d.npy', X_2d); np.save(f'{OUTPUT_DIR}/y.npy', y)
np.save(f'{OUTPUT_DIR}/K_ii.npy', K_ii); np.save(f'{OUTPUT_DIR}/rhs.npy', rhs)
np.save(f'{OUTPUT_DIR}/internal_mask.npy', internal)
rmse = np.sqrt(np.mean((u_quantum_i - u_ref)**2))
meta = {'method':'energy','n_internal':int(n_i),'qubo_size':int(nvar),'bit_width':BIT_WIDTH,'rmse_vs_classical':float(rmse),'best_energy':float(energies.min())}
with open(f'{OUTPUT_DIR}/meta_energy.json','w') as f: json.dump(meta, f, indent=2)
print(f'预设已保存到 {OUTPUT_DIR}, RMSE vs 经典: {rmse:.6e}')

预设已保存到 D:/QPDE/pde+pinn/outputs_bc, RMSE vs 经典: 3.888150e-01
